# EDA on cleaned data

Analysis of `data/data_cleaned.csv` produced by `clean-feature-engineer.py`.
Focus: feature distributions after cleaning, feature-severity relationships relevant
to modeling, class imbalance, and feature selection rationale.

Casualty columns were dropped from this dataset because cross-tab analysis showed
their severity distribution is nearly identical across all category values (see
EDA_raw.ipynb for the raw cross-tab).

Color convention: Fatal injury = red, Serious Injury = yellow, Slight Injury = green.


## Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

SEVERITY_ORDER = ["Slight Injury", "Serious Injury", "Fatal injury"]
SEVERITY_COLORS = {
    "Slight Injury":  "#2ca02c",
    "Serious Injury": "#f1c40f",
    "Fatal injury":   "#d62728",
}
SEVERITY_PALETTE = [SEVERITY_COLORS[s] for s in SEVERITY_ORDER]


In [ ]:
def severity_crosstab(s, target):
    pivot = pd.crosstab(s, target)
    for col in SEVERITY_ORDER:
        if col not in pivot.columns:
            pivot[col] = 0
    return pivot[SEVERITY_ORDER]


def plot_count_and_pct(series, target, title, order=None, rotation=0, figsize=(12, 4)):
    pivot = severity_crosstab(series, target)
    if order is not None:
        pivot = pivot.reindex([o for o in order if o in pivot.index])

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    pivot.plot(kind="bar", stacked=True, color=SEVERITY_PALETTE, ax=axes[0])
    axes[0].set_title(f"{title} — count")
    axes[0].set_ylabel("Number of accidents")
    axes[0].tick_params(axis="x", rotation=rotation)
    axes[0].grid(axis="y", linestyle="--", alpha=0.4)

    pct = pivot.div(pivot.sum(axis=1).replace(0, np.nan), axis=0)
    pct.plot(kind="bar", stacked=True, color=SEVERITY_PALETTE, ax=axes[1])
    axes[1].set_title(f"{title} — percentage")
    axes[1].set_ylabel("Share within group")
    axes[1].set_ylim(0, 1)
    axes[1].tick_params(axis="x", rotation=rotation)
    axes[1].grid(axis="y", linestyle="--", alpha=0.4)

    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].get_legend().remove()
    axes[1].legend(handles, labels, title="Severity", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


## Load and overview

In [ ]:
df = pd.read_csv("data/data_cleaned.csv")
target = df["Accident_severity"]
print("shape:", df.shape)
print("NaN anywhere:", df.isna().sum().sum())
df.head(3)

In [ ]:
# Full column summary after pipeline.
summary = []
for col in df.columns:
    s = df[col]
    if s.dtype.kind in ("i", "f"):
        rng = f"min={s.min()}, max={s.max()}, mean={s.mean():.2f}"
    else:
        top = ", ".join(s.value_counts().head(4).index.astype(str))
        rng = top
    summary.append({"column": col, "dtype": str(s.dtype),
                    "n_unique": s.nunique(), "sample": rng})
pd.DataFrame(summary)

## Target distribution and class imbalance

Fatal injury is 1.3% of the data — about 65x rarer than Slight Injury.
Any classifier trained without imbalance handling will almost certainly ignore the
Fatal class entirely. Recommended strategy: SMOTE on the training fold only
(never on the full dataset before splitting).


In [ ]:
counts = target.value_counts().reindex(SEVERITY_ORDER)
pct = (counts / counts.sum() * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
counts.plot(kind="bar", color=SEVERITY_PALETTE, ax=axes[0])
axes[0].set_title("Count")
axes[0].tick_params(axis="x", rotation=0)
for i, v in enumerate(counts):
    axes[0].text(i, v, str(v), ha="center", va="bottom")

pct.plot(kind="bar", color=SEVERITY_PALETTE, ax=axes[1])
axes[1].set_title("Percentage")
axes[1].tick_params(axis="x", rotation=0)
for i, v in enumerate(pct):
    axes[1].text(i, v, f"{v}%", ha="center", va="bottom")

plt.tight_layout()
plt.show()

imbalance_ratio = counts.max() / counts.min()
print(f"Imbalance ratio (Slight / Fatal): {imbalance_ratio:.0f}x")

## Feature-severity relationships

Percentage plots are more useful than count plots here — Fatal is so rare that
it is invisible on a count scale. The percentage view shows which categories
shift the Fatal/Serious share most.


### Time

In [ ]:
plot_count_and_pct(df["Hour"], target, "Hour of day", figsize=(14, 4))

In [ ]:
plot_count_and_pct(df["TimeOfDay"], target, "Time of day",
                   order=["Night", "Morning", "Afternoon", "Evening"])

In [ ]:
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
plot_count_and_pct(df["Day_of_week"], target, "Day of week", order=DAY_ORDER)

### Driver

In [ ]:
plot_count_and_pct(df["Age_band_of_driver"], target, "Age band of driver",
                   order=["Under 18", "18-30", "31-50", "Over 51", "Unknown"])

In [ ]:
EXP_ORDER = ["No Licence", "Below 1yr", "1-2yr", "2-5yr", "5-10yr", "Above 10yr", "Unknown"]
plot_count_and_pct(df["Driving_experience"], target,
                   "Driving experience", order=EXP_ORDER, rotation=20)

In [ ]:
EDU_ORDER = ["Illiterate", "Writing & reading", "Elementary school",
             "Junior high school", "High school", "Above high school", "Unknown"]
plot_count_and_pct(df["Educational_level"], target,
                   "Educational level", order=EDU_ORDER, rotation=25)

### Vehicle and road

In [ ]:
top_types = df["Type_of_vehicle"].value_counts().head(10).index
plot_count_and_pct(
    df["Type_of_vehicle"].where(df["Type_of_vehicle"].isin(top_types), "Other/rare"),
    target, "Type of vehicle (top 10)", rotation=30, figsize=(14, 4))

In [ ]:
plot_count_and_pct(df["Light_conditions"], target, "Light conditions", rotation=20)

In [ ]:
plot_count_and_pct(df["Weather_conditions"], target, "Weather conditions", rotation=30)

In [ ]:
plot_count_and_pct(df["Road_surface_conditions"], target, "Road surface conditions")

In [ ]:
plot_count_and_pct(df["Types_of_Junction"], target, "Junction type", rotation=20)

### Collision

In [ ]:
plot_count_and_pct(df["Type_of_collision"], target,
                   "Type of collision", rotation=30, figsize=(14, 4))

In [ ]:
cause_top = df["Cause_of_accident"].value_counts().head(12).index
plot_count_and_pct(
    df["Cause_of_accident"].where(df["Cause_of_accident"].isin(cause_top), "Other/rare"),
    target, "Cause of accident (top 12)", rotation=45, figsize=(14, 5))

## Numeric feature distributions

In [ ]:
# Hour distribution by severity — violin or box is better than a bar here.
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, sev, color in zip(axes, SEVERITY_ORDER, SEVERITY_PALETTE):
    mask = target == sev
    axes_list = df.loc[mask, "Hour"]
    ax.hist(axes_list, bins=24, range=(0, 24), color=color, edgecolor="white")
    ax.set_title(sev)
    ax.set_xlabel("Hour")
    ax.set_ylabel("Count")
plt.suptitle("Hour distribution per severity class")
plt.tight_layout()
plt.show()

In [ ]:
plot_count_and_pct(df["Number_of_vehicles_involved"], target,
                   "Number of vehicles involved")

In [ ]:
plot_count_and_pct(df["Number_of_casualties"], target,
                   "Number of casualties")

## Ordinal engineered features

The `_ord` columns encode ordered categories as integers (-1 = Unknown/not on scale).
These give models a numeric handle on ordinal relationships that OHE cannot capture.


In [ ]:
ord_cols = [c for c in df.columns if c.endswith("_ord")]
fig, axes = plt.subplots(1, len(ord_cols), figsize=(16, 4))
for ax, col in zip(axes, ord_cols):
    df[col].value_counts().sort_index().plot(kind="bar", ax=ax, color="#555")
    ax.set_title(col.replace("_ord", ""))
    ax.set_xlabel("ordinal value")
    ax.tick_params(axis="x", rotation=0)
plt.suptitle("Ordinal feature distributions  (-1 = Unknown)")
plt.tight_layout()
plt.show()

In [ ]:
# Ordinal value vs severity — does the ordinal ranking track risk?
for col in ord_cols:
    plot_count_and_pct(df[col], target, col.replace("_ord", "") + " (ordinal)")

## Correlation among numeric features

Only numeric columns (including ordinal codes). Useful for spotting redundancy
before model selection.


In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
# Exclude the target code itself from the correlation heatmap inputs.
corr_cols = [c for c in num_cols if c != "Severity_code"]

corr = df[corr_cols + ["Severity_code"]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(corr.columns, fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.03)
ax.set_title("Correlation matrix (numeric + ordinal features)")
plt.tight_layout()
plt.show()

# Top correlations with Severity_code.
sev_corr = corr["Severity_code"].drop("Severity_code").sort_values(key=abs, ascending=False)
print("Correlations with Severity_code:")
print(sev_corr.round(3))

## Feature set for modeling

Columns in `data/data_cleaned.csv` and their intended role:


In [ ]:
# Print the feature contract: name, type, intended use.
cat_cols = df.select_dtypes(include="object").columns.tolist()
cat_cols = [c for c in cat_cols if c not in ("Accident_severity",)]
int_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
int_cols = [c for c in int_cols if c not in ("Severity_code",)]

print(f"Target: Accident_severity  (string) / Severity_code (int 0-2)")
print()
print(f"Categorical features ({len(cat_cols)}):")
for c in cat_cols:
    print(f"  {c}  ({df[c].nunique()} levels)")
print()
print(f"Numeric / ordinal features ({len(int_cols)}):")
for c in int_cols:
    print(f"  {c}  (min={df[c].min()}, max={df[c].max()})")

## Modeling notes

- **Class imbalance**: apply SMOTE on the training fold only, after train-test split.
- **Categorical encoding**: TabNet needs OrdinalEncoder + `cat_idxs`/`cat_dims`.
  Stacking (XGBoost + ExtraTrees) works with OrdinalEncoder directly.
  MLP needs OHE or learned embeddings.
- **Ordinal columns (`_ord`)**: can be fed as numeric to any model.
  The -1 sentinel is meaningful — it separates 'not on the scale' from the
  lowest ordinal value (0).
- **`Unknown` category**: treat as its own level in all encoders, not as NaN.
